### Helper functions we'll use to parse out the data

In [123]:
## a function to find the distance between the guide and target
def edit_distance(seq1, seq2):
    return(sum([0 if x == y else 1 for x,y in zip(seq1,seq2)]))

print(edit_distance("GTCTAGGTTTAAACCTAGAC","GTCTAGGTTTAAACCTAGAC"))
print(edit_distance("GTCTAGGTTTAAACCTAGAC","GTCTAGGTTTAAACCTATTC"))

0
2


In [124]:
### Load up a GESTALT statistics file, where we've encoded the guide and the target as a 
def load_GESTALT_statistics_file(input_file, known_guide_dictionary):
    input_fl = open(input_file)
    input_fl.readline() # read off the header

    # store our guide->target relationship and how many of the known targets we've seen
    guide_targets = {}
    known_matches = {key:False for (key,value) in known_targets.items()}
    
    for line in input_fl:
        if "MERGED" in line:
            sp = line.strip("\n").split("\t")
            guide = sp[24][0:20]
            target = sp[25][3:23]

            matched_guide = "NONE"
            if guide in known_targets:
                matched_guide = guide
                known_matches[guide] = True

            if not guide in guide_targets:
                guide_targets[guide] = []
            stored_guides = guide_targets[guide]
            stored_guides.append(target)
            guide_targets[guide] = stored_guides

    return ((guide_targets,known_matches))

In [125]:
def contingency_table(guide_target_dict,known_targets_dict):
    # count the size of the reads that match the known sequences vs error sequences
    known_guide_target_match = 0
    known_guide_target_mismatch = 0
    unknown_guide_target_match = 0
    unknown_guide_target_mismatch = 0

    for guide,target_array in guide_target_dict.items():
        target_guide_matches = sum([1 if x == guide else 0 for x in target_array])
        if guide in known_targets_dict:
            known_guide_target_match += target_guide_matches
            known_guide_target_mismatch += len(target_array) - target_guide_matches
        else:
            unknown_guide_target_match += target_guide_matches
            unknown_guide_target_mismatch += len(target_array) - target_guide_matches

    return((known_guide_target_match,
            known_guide_target_mismatch,
            unknown_guide_target_match,
            unknown_guide_target_mismatch))


In [126]:
# print out stats about the library matchups
def print_stats(contingency_table,samples,lib):
    print("total number of filtered, " + lib +  " 'guide-target' UMIs seen " + str(sum(contingency_table)) + "\n")

    match_pct = str(100.0 * float(contingency_table[0])/(float(contingency_table[0])+ float(contingency_table[1])))
    print("known " + lib +  " guide, how many targets match: " + str(contingency_table[0]) + " %= " + match_pct)
    print("known " + lib +  " guide, how many targets mismatch: " + str(contingency_table[1]))
    print("Total UMI with known guide count: " + str(contingency_table[0] + contingency_table[1]))
    print("Of the programmed guides, how many did we see: " + str(sum(samples[1].values())) + "\n")

    match_pct = str(100.0 * float(wt_contingency_table[2])/(float(contingency_table[2])+ float(contingency_table[3])))
    print("unknown " + lib +  " guide, how many targets match: " + str(contingency_table[2]) + " %= " + match_pct)
    print("unknown " + lib +  " guide, how many targets mismatch: " + str(contingency_table[3]))
    print("Total UMI with unknown guide count: " + str(contingency_table[2] + contingency_table[3]))

In [127]:
from enum import Enum
class EditOutcome(Enum):
    NONE = 0
    LEFT = 1
    RIGHT = 2
    BOTH = 3
    BAD = 4

# some tools to help us look for base editing changes

def edit_distance_no_dash(seq1, seq2):
    return(sum([0 if x == y and x != '-' else 1 for x,y in zip(seq1,seq2)]))

def sequence_differences(str1,str2):
    return(list(filter(lambda s: s[0] != s[1], [(x,y,i) for i,(x,y) in enumerate(zip(str1,str2))])))

base_editing_windows = [(2,8), (12,18)]
abe_conversions_over_windows = [("A","G"),("T","C")]
cbe_conversions_over_windows = [("C","T"),("G","A")]

def windowed_edit_counts(reference,read,conversions,windows):
    edits_per_window = [0 for x in windows]
    
    for ref,read,position in sequence_differences(reference,read):
        for window_index,(window,be_pairs) in enumerate(zip(windows,conversions)):
            if position >= window[0] and position < window[1] and ref == be_pairs[0] and read == be_pairs[1]:
                edits_per_window[window_index] += 1
        
    return(edits_per_window)

def indel_proportion(sequence1,sequence2):
    return(sum([1 if (x == '-' or y == '-') else 0 for x,y in zip(sequence1,sequence2)]))

def edit_windows_to_call(windows):
    # we assume a left and a right here
    assert(len(windows) == 2)
    if windows[0] == 0 and windows[1] == 0:
        return EditOutcome.NONE
    elif windows[0] > 0 and windows[1] == 0:
        return EditOutcome.LEFT
    elif windows[0] == 0 and windows[1] > 0:
        return EditOutcome.RIGHT
    return EditOutcome.BOTH
    
# 0 - no editing, 1 - left, 2 - right, 3 - both, 4 - bad
def call_editing(guide,target,max_mismatches,is_abe):
    if edit_distance_no_dash(guide,target) > max_mismatches:
        return EditOutcome.BAD
    
    if is_abe:
        return(edit_windows_to_call(windowed_edit_counts(guide,target,abe_conversions_over_windows,base_editing_windows)))
    else:
        return(edit_windows_to_call(windowed_edit_counts(guide,target,cbe_conversions_over_windows,base_editing_windows)))


# test out a couple combinations for ABE
print(call_editing("GTCTAGGTTTAAACCTAGAC","GTCTAGGTTTAAACCTAGAC",5,True))
print(call_editing("GTCTAGGTTTAAACCTAGAC","GTCTAGGTTTAAACCCAGAC",5,True))
print(call_editing("GTCTAGGTTTAAACCTAGAC","GTCTGGGTTTAAACCTAGAC",5,True))
print(call_editing("GTCTAGGTTTAAACCTAGAC","GTCTGGGTTTAAACCCAGAC",5,True))
print(call_editing("GTCTAGGTTTAAACCTAGAC","GTC----------------C",5,True))

# test out a couple combinations for CBE
print(call_editing("GTCTAGGTTTAAACCTAGAC","GTTTAGGTTTAAACCTAGAC",5,False))
print(call_editing("GTCTAGGTTTAAACCTAGAC","GTCTAGGTTTAAACCTAAAC",5,False))
print(call_editing("GTCTAGGTTTAAACCTAGAC","GTTTAGGTTTAAACCTAAAC",5,False))

# a function that tallies up outcomes for pairs of guide-targets
def tally_events(guide_to_target_array,known_guides,stats_file,lib,is_abe,max_mismatches=6):
    total_combinations = [0,0,0,0,0] # NONE, LEFT, RIGHT, BOTH, BAD

    for guide,target_array in guide_to_target_array.items():
        is_known = guide in known_guides
        guide_combinations = [0,0,0,0,0] # NONE, LEFT, RIGHT, BOTH, BAD
        for target in target_array:
            guide_combinations[call_editing(guide,target,max_mismatches,is_abe).value] += 1

        stats_file.write(lib + "\t" + guide + "\t" + str(is_known) + "\t" + str(len(target_array)) + "\t" + "\t".join([str(x) for x in guide_combinations]) + "\n")
        total_combinations = [x + y for x,y in zip(total_combinations,guide_combinations)]
    return(total_combinations)



EditOutcome.NONE
EditOutcome.RIGHT
EditOutcome.LEFT
EditOutcome.BOTH
EditOutcome.BAD
EditOutcome.LEFT
EditOutcome.RIGHT
EditOutcome.BOTH


## Load up the known target list and extract the guide sequence

In [128]:
known_targets_file = open("../../2019_11_27_Maryam_palendromes/synth_constructs_2018_08_28.fasta")

known_targets = {}

for line in known_targets_file:
    if ">" in line:
        target = line.split("_")[1][0:20]
        known_targets[target] = known_targets.get(target,0) + 1

print("We have " + str(len(known_targets)) + " unique, known targets")

We have 314 unique, known targets


## A file we'll use to store all of the guide -> target editing outcomes

In [129]:
tally_output = open("tallied_outcomes.tsv","w")
tally_output.write("lib\tguide\tisKnown\ttargets\tnone\tleft\tright\tboth\tbad\n")

51

## Process the normal (WT) samples

In [130]:
wt_samples = load_GESTALT_statistics_file("../data/stats/pigpallib_S7_R1.stats", known_targets)
wt_contingency_table = contingency_table(wt_samples[0],known_targets)

print_stats(wt_contingency_table,wt_samples,"WT")
tally_events(wt_samples[0],known_targets,tally_output,"WT",True)

total number of filtered, WT 'guide-target' UMIs seen 5414

known WT guide, how many targets match: 659 %= 29.792043399638338
known WT guide, how many targets mismatch: 1553
Total UMI with known guide count: 2212
Of the programmed guides, how many did we see: 262

unknown WT guide, how many targets match: 674 %= 21.049344159900063
unknown WT guide, how many targets mismatch: 2528
Total UMI with unknown guide count: 3202


[2244, 100, 64, 13, 2993]

## Process the ABE samples

In [131]:
abe_samples = load_GESTALT_statistics_file("../data/stats/ABpigpallib_S8_R1.stats", known_targets)
abe_contingency_table = contingency_table(abe_samples[0],known_targets)

print_stats(abe_contingency_table,abe_samples,"ABE")
tally_events(abe_samples[0],known_targets,tally_output,"ABE",True)

total number of filtered, ABE 'guide-target' UMIs seen 7418

known ABE guide, how many targets match: 1354 %= 36.164529914529915
known ABE guide, how many targets mismatch: 2390
Total UMI with known guide count: 3744
Of the programmed guides, how many did we see: 226

unknown ABE guide, how many targets match: 852 %= 18.34512792596625
unknown ABE guide, how many targets mismatch: 2822
Total UMI with unknown guide count: 3674


[3299, 133, 119, 28, 3839]

## Process the CBE samples

In [132]:
cbe_samples = load_GESTALT_statistics_file("../data/stats/BEpigpallib_S9_R1.stats", known_targets)
cbe_contingency_table = contingency_table(cbe_samples[0],known_targets)

print_stats(cbe_contingency_table,cbe_samples,"CBE")
tally_events(cbe_samples[0],known_targets,tally_output,"CBE",False)

total number of filtered, CBE 'guide-target' UMIs seen 6155

known CBE guide, how many targets match: 1147 %= 40.61614730878187
known CBE guide, how many targets mismatch: 1677
Total UMI with known guide count: 2824
Of the programmed guides, how many did we see: 205

unknown CBE guide, how many targets match: 560 %= 20.234163914740318
unknown CBE guide, how many targets mismatch: 2771
Total UMI with unknown guide count: 3331


[2952, 66, 19, 10, 3108]

In [133]:
tally_output.close()